# Feature Engineering on High Volume For-Hire Vehicle (HVFHV) Trip Records Dataset:

In this notebook, we are mainly focusing on creating new spark dataframes about the total revenue in different time series for drivers in HVFHV dataset.

----

# Import Libraries:

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql import functions as F
from pyspark.sql.functions import when, col, count
from pyspark.sql.functions import sum as spark_sum
import os

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_hvfhv_revenue")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

24/08/21 01:24:09 WARN Utils: Your hostname, Cocos-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.16.33.67 instead (on interface en0)
24/08/21 01:24:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/21 01:24:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Read Files:

In [3]:
base_dir = "../data"
hvfhv_path = base_dir + '/developed/merged_data/full_hvfhv'
hvfhv_sdf = spark.read.parquet(hvfhv_path)
hvfhv_sdf.show(5)

+-----------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+-------------------------+-------------------+-----------------+------------------+-----------+-----------+------------+------------+-----+
|hvfhs_license_num|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|day_of_week|request_to_pickup_minutes|         trip_speed|total_fare_amount|     total_revenue|pickup_date|pickup_hour|dropoff_date|dropoff_hour|month|
+-----------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+------

In [4]:
hvfhv_sdf = hvfhv_sdf.withColumn(
    "day_type",
    when(hvfhv_sdf["day_of_week"].isin(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]), "Weekday")
    .otherwise("Weekend")
)

hvfhv_sdf.show(5)

+-----------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+-------------------------+-------------------+-----------------+------------------+-----------+-----------+------------+------------+-----+--------+
|hvfhs_license_num|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|day_of_week|request_to_pickup_minutes|         trip_speed|total_fare_amount|     total_revenue|pickup_date|pickup_hour|dropoff_date|dropoff_hour|month|day_type|
+-----------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+

In [5]:
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 100725779
Number of columns: 28


In [6]:
hvfhv_sdf.printSchema()

root
 |-- hvfhs_license_num: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: double (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: integer (nullable = true)
 |-- shared_match_flag: integer (nullable = true)
 |-- wav_request_flag: integer (nullable = true)
 |-- wav_match_flag: integer (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- request_to_pickup_minutes: double (nullable = true)
 |-- trip_speed: double (nullable = true)
 |-- total_fare_amount: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 

# Hourly Revenue:

Since driver pay is generated after passengers place the order online, it is more accurate to use pickup hour not dropoff hour.

In [7]:
TOTAL_HOURS = 24

# Group by `pickup_hour` `pickup_date``, `day_type`,
# then sum the driver revenue for each group
hourly_revenue_sdf = hvfhv_sdf.groupBy( "pickup_hour", "pickup_date", "day_type") \
                              .agg(spark_sum("total_revenue").alias("hourly_revenue")) \
                              .orderBy("hourly_revenue")

# Standardize the daily revenue
hourly_revenue_sdf = hourly_revenue_sdf.withColumn('mean_hourly_revenue',
                                                   col('hourly_revenue') / TOTAL_HOURS)

hourly_revenue_sdf = hourly_revenue_sdf.orderBy("pickup_hour")\
                                       .drop("hourly_revenue")

hourly_revenue_sdf.show(5)

+-----------+-----------+--------+-------------------+
|pickup_hour|pickup_date|day_type|mean_hourly_revenue|
+-----------+-----------+--------+-------------------+
|          0| 2023-08-19| Weekend|  24378.10624999972|
|          0| 2023-08-24| Weekday| 11312.753333333325|
|          0| 2023-08-27| Weekend|  28295.61124999968|
|          0| 2023-08-20| Weekend|  27160.79041666658|
|          0| 2023-08-26| Weekend| 23362.556250000005|
+-----------+-----------+--------+-------------------+
only showing top 5 rows



In [8]:
hourly_revenue_sdf.describe().show()

+-------+-----------------+--------+-------------------+
|summary|      pickup_hour|day_type|mean_hourly_revenue|
+-------+-----------------+--------+-------------------+
|  count|             4416|    4416|               4416|
|   mean|             11.5|    NULL| 20088.299765153282|
| stddev|6.922970447632971|    NULL|  8537.909093894414|
|    min|                0| Weekday|  2811.679999999998|
|    max|               23| Weekend|  64694.53083333312|
+-------+-----------------+--------+-------------------+



# Hourly Revenue Among Day of Week:

In [9]:
# Group by `day_of_week`, "day_type", "pickup_date", "pickup_hour" 
# then sum the driver revenue for each group
hourly_revenue_among_day_of_week = hvfhv_sdf.groupBy("pickup_date", "day_of_week",
                                            "day_type", "pickup_hour") \
                                   .agg(spark_sum("total_revenue").alias("day_by_hour_revenue")) \
                                   .orderBy("day_of_week")

# Standardize the hourly revenue
hourly_revenue_among_day_of_week = hourly_revenue_among_day_of_week.withColumn('day_by_hour_revenue',
                                                                               col('day_by_hour_revenue') / TOTAL_HOURS)

hourly_revenue_among_day_of_week.show(5)

+-----------+-----------+--------+-----------+-------------------+
|pickup_date|day_of_week|day_type|pickup_hour|day_by_hour_revenue|
+-----------+-----------+--------+-----------+-------------------+
| 2023-08-18|     Friday| Weekday|         18| 28010.772916666672|
| 2023-08-25|     Friday| Weekday|         21| 23225.642083333227|
| 2023-08-18|     Friday| Weekday|         22|  27467.40458333305|
| 2023-08-25|     Friday| Weekday|         20| 24169.993333333234|
| 2023-08-25|     Friday| Weekday|          6| 14905.668749999952|
+-----------+-----------+--------+-----------+-------------------+
only showing top 5 rows



In [10]:
hourly_revenue_among_day_of_week.describe().show()

+-------+-----------+--------+-----------------+-------------------+
|summary|day_of_week|day_type|      pickup_hour|day_by_hour_revenue|
+-------+-----------+--------+-----------------+-------------------+
|  count|       4416|    4416|             4416|               4416|
|   mean|       NULL|    NULL|             11.5| 20088.299765153217|
| stddev|       NULL|    NULL|6.922970447632987|  8537.909093894403|
|    min|     Friday| Weekday|                0|  2811.679999999998|
|    max|  Wednesday| Weekend|               23|  64694.53083333312|
+-------+-----------+--------+-----------------+-------------------+



# Daily Revenue:

In [11]:
TOTAL_DAYS = 31 + 31 + 30 + 31 + 30 + 31

# Group by `pickup_date` and `PULocationID`, then sum the revenue for each group
daily_revenue_sdf = hvfhv_sdf.groupBy("pickup_date", "PULocationID") \
                             .agg(spark_sum("total_revenue").alias("daily_revenue")) \
                             .orderBy("pickup_date", "PULocationID")

# Standardize the daily revenue
daily_revenue_sdf = daily_revenue_sdf.withColumn('daily_revenue', 
                                                 col('daily_revenue') / TOTAL_DAYS)

daily_revenue_sdf.show(5)

+-----------+------------+-------------------+
|pickup_date|PULocationID|      daily_revenue|
+-----------+------------+-------------------+
| 2023-07-01|           2|0.24369565217391306|
| 2023-07-01|           3|  102.3845652173913|
| 2023-07-01|           4|  230.4214130434783|
| 2023-07-01|           5|  19.35326086956521|
| 2023-07-01|           6|  34.76135869565217|
+-----------+------------+-------------------+
only showing top 5 rows



In [12]:
daily_revenue_sdf.describe().show()

+-------+-----------------+--------------------+
|summary|     PULocationID|       daily_revenue|
+-------+-----------------+--------------------+
|  count|            47387|               47387|
|   mean|132.8339206955494|   244.1779531248717|
| stddev|75.97943748101534|    294.513008223253|
|    min|                1|0.029293478260869563|
|    max|              263|   4643.692880434764|
+-------+-----------------+--------------------+



# Monthly Revenue:

Extract monthly revenue in each location ID:

In [13]:
TOTAL_MONTHS = 6 # the timeline is 6 month

# Group by `month`, 
# then sum the driver revenue for each group
monthly_revenue_sdf = hvfhv_sdf.groupBy("month") \
                               .agg(spark_sum("total_revenue").alias("monthly_revenue")) \
                               .orderBy("month")

# Standardize the monthly revenue
monthly_revenue_sdf = monthly_revenue_sdf.withColumn('monthly_revenue', 
                                                     col('monthly_revenue') / TOTAL_MONTHS)

monthly_revenue_sdf.show(5)

+-----+-------------------+
|month|    monthly_revenue|
+-----+-------------------+
|    7|5.544857142832748E7|
|    8| 5.35656964866589E7|
|    9|6.280313457999557E7|
|   10|6.233148135499249E7|
|   11|5.813192801333028E7|
+-----+-------------------+
only showing top 5 rows



In [14]:
monthly_revenue_sdf.describe().show()

+-------+------------------+--------------------+
|summary|             month|     monthly_revenue|
+-------+------------------+--------------------+
|  count|                 6|                   6|
|   mean|               9.5|5.9139954508606635E7|
| stddev|1.8708286933869707|  4025141.0641603763|
|    min|                 7|  5.35656964866589E7|
|    max|                12| 6.280313457999557E7|
+-------+------------------+--------------------+



# Save the Merged Datasets:

Save the hourly revenue dataset:

In [15]:
hour_dir = base_dir + '/developed/merged_data'
file_name = 'hourly_revenue'
hour_path = os.path.join(hour_dir, file_name)
hourly_revenue_sdf.write.mode('overwrite').parquet(hour_path)

Save the hourly revenue among day of week dataset:

In [16]:
hourly_revenue_among_day_of_week_dir = base_dir + '/developed/merged_data'
file_name = 'hourly_revenue_among_day_of_week'
hourly_revenue_among_day_of_week_path = os.path.join(hourly_revenue_among_day_of_week_dir, file_name)
hourly_revenue_among_day_of_week.write.mode('overwrite').parquet(hourly_revenue_among_day_of_week_path)

Save the daily revenue dataset:

In [17]:
day_dir = base_dir + '/developed/merged_data'
file_name = 'daily_revenue'
day_path = os.path.join(day_dir, file_name)
daily_revenue_sdf.write.mode('overwrite').parquet(day_path)

Save the monthly revenue dataset:

In [18]:
month_dir = base_dir + '/developed/merged_data'
file_name = 'monthly_revenue'
month_path = os.path.join(month_dir, file_name)
monthly_revenue_sdf.write.mode('overwrite').parquet(month_path)